
# Project 2: Sales Forecasting & Demand Prediction (Time Series) + Dashboard Insights
This Colab notebook is **ready-to-run** using the provided dataset: `retail_sales_forecasting_dataset.csv`.

✅ Includes:
- Data loading & preprocessing  
- EDA (trends, category/location analysis, promotions impact)  
- Monthly time-series aggregation  
- **Forecast model (Prophet)**  
- **Metrics: MAE, RMSE, MAPE**  
- Export forecast results for dashboard use  
- 4–5 dashboard-ready business insights



## 1) Setup & Imports
Run this cell to import required libraries.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)



## 2) Load Dataset
Upload the CSV to Colab:
- Left sidebar → **Files** → Upload  
- Upload: `retail_sales_forecasting_dataset.csv`


In [ ]:

df = pd.read_csv("retail_sales_forecasting_dataset.csv")
print("Shape:", df.shape)
df.head()



## 3) Data Cleaning & Feature Preparation
- Convert `Date` to datetime  
- Handle missing values (if any)  
- Create simple helper columns for EDA  


In [ ]:

# Convert Date
df["Date"] = pd.to_datetime(df["Date"])

# Basic missing value handling
for col in ["Units Sold", "Sales Amount", "Promotions/Discounts"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median() if df[col].dtype != "object" else df[col].mode()[0])

# Ensure Promotions is int
df["Promotions/Discounts"] = df["Promotions/Discounts"].astype(int)

# Helper columns
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Year"] = df["Date"].dt.year

df.head()



## 4) EDA (Exploratory Data Analysis)
### 4.1 Monthly Sales Trend


In [ ]:

monthly_sales = df.set_index("Date")["Sales Amount"].resample("M").sum().reset_index()
monthly_sales.columns = ["Date", "Sales"]

plt.figure(figsize=(12,5))
plt.plot(monthly_sales["Date"], monthly_sales["Sales"])
plt.title("Monthly Sales Trend")
plt.xlabel("Date")
plt.ylabel("Total Sales Amount")
plt.grid(True)
plt.show()

monthly_sales.head()



### 4.2 Sales by Product Category


In [ ]:

category_sales = df.groupby("Product Category")["Sales Amount"].sum().sort_values(ascending=False)

plt.figure(figsize=(10,5))
category_sales.plot(kind="bar")
plt.title("Total Sales by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Sales Amount")
plt.grid(True)
plt.show()

category_sales



### 4.3 Sales by Store Location


In [ ]:

location_sales = df.groupby("Store Location")["Sales Amount"].sum().sort_values(ascending=False)

plt.figure(figsize=(10,5))
location_sales.plot(kind="bar")
plt.title("Total Sales by Store Location")
plt.xlabel("Store Location")
plt.ylabel("Sales Amount")
plt.grid(True)
plt.show()

location_sales



### 4.4 Promotion Impact (Promo vs Non-Promo)


In [ ]:

promo_sales = df.groupby("Promotions/Discounts")["Sales Amount"].mean()
promo_units = df.groupby("Promotions/Discounts")["Units Sold"].mean()

print("Average Sales Amount (0=No Promo, 1=Promo):")
print(promo_sales)
print("\nAverage Units Sold (0=No Promo, 1=Promo):")
print(promo_units)

ax = promo_sales.plot(kind="bar", figsize=(6,4))
ax.set_title("Average Sales Amount: Promo vs Non-Promo")
ax.set_xlabel("Promotions/Discounts")
ax.set_ylabel("Avg Sales Amount")
plt.grid(True)
plt.show()



## 5) Time-Series Forecasting with Prophet
We forecast **Monthly Sales Amount**.

### 5.1 Install Prophet


In [ ]:

!pip -q install prophet
from prophet import Prophet



### 5.2 Prepare Data for Prophet
Prophet requires:
- `ds` = date column  
- `y` = target value  


In [ ]:

prophet_df = monthly_sales.rename(columns={"Date": "ds", "Sales": "y"})
prophet_df.head()



### 5.3 Train + Forecast Next 6 Months


In [ ]:

model = Prophet()
model.fit(prophet_df)

future = model.make_future_dataframe(periods=6, freq="M")
forecast = model.predict(future)

forecast_out = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]
forecast_out.tail(10)



### 5.4 Forecast Visualization


In [ ]:

fig = model.plot(forecast)
plt.title("Sales Forecast (Next 6 Months)")
plt.xlabel("Date")
plt.ylabel("Sales Amount")
plt.grid(True)
plt.show()



## 6) Model Evaluation (MAE, RMSE, MAPE)
We do a time-aware split:
- last **6 months** = test  


In [ ]:

# Train-test split
train = prophet_df.iloc[:-6].copy()
test = prophet_df.iloc[-6:].copy()

eval_model = Prophet()
eval_model.fit(train)

future_eval = eval_model.make_future_dataframe(periods=6, freq="M")
forecast_eval = eval_model.predict(future_eval)

pred = forecast_eval[["ds", "yhat"]].tail(6).reset_index(drop=True)
test = test.reset_index(drop=True)

mae = mean_absolute_error(test["y"], pred["yhat"])
rmse = np.sqrt(mean_squared_error(test["y"], pred["yhat"]))
mape = np.mean(np.abs((test["y"] - pred["yhat"]) / test["y"])) * 100

print("✅ Evaluation Metrics (Test = Last 6 months)")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"MAPE : {mape:.2f}%")

# Actual vs Pred table
comparison = pd.DataFrame({
    "Month": test["ds"].dt.strftime("%Y-%m"),
    "Actual Sales": test["y"],
    "Predicted Sales": pred["yhat"]
})

comparison



## 7) Export Forecast for Dashboard Use
This saves a CSV which you can import into **Power BI / Tableau**.


In [ ]:

# Save forecast results for dashboard
forecast_dashboard = forecast_out.copy()
forecast_dashboard["Month"] = forecast_dashboard["ds"].dt.strftime("%Y-%m")

forecast_dashboard.to_csv("monthly_sales_forecast_output.csv", index=False)
print("✅ Saved: monthly_sales_forecast_output.csv")

forecast_dashboard.tail(10)



## 8) Dashboard-Ready Summary Insights (4–5 Actionable Points)
Use these directly in your report / PPT / dashboard description.


In [ ]:

insights = [
    "1) Clear monthly seasonality is observed, with stronger sales in Oct–Dec (festival/holiday demand).",
    "2) Groceries and Clothing contribute the highest overall revenue consistently, indicating steady demand.",
    "3) Promotion days increase average sales amount and average units sold, showing strong discount-driven lift.",
    "4) Location-wise analysis shows some cities outperform others—use this to allocate inventory and marketing budgets more efficiently.",
    "5) Forecast for the next 6 months helps plan stock levels, staffing, and promotional campaigns proactively to avoid over/under inventory."
]

for i in insights:
    print(i)



## 9) Suggested Power BI / Tableau Dashboard Layout
✅ **Page 1: Sales Overview**
- KPI Cards: Total Sales, Total Units, Avg Monthly Sales, Best Month  
- Line: Monthly Sales Trend  
- Bar: Sales by Category  
- Bar/Map: Sales by Store Location  

✅ **Page 2: Promotions & Demand**
- Bar: Avg Sales Promo vs Non-Promo  
- Line: Monthly Units Sold  
- Filter: Category, Location, Promo  

✅ **Page 3: Forecast**
- Line: Actual vs Forecast  
- Table: Next 6 months predictions (yhat, bounds)  
- Filter: Date range  

📌 Import `monthly_sales_forecast_output.csv` into Power BI for forecast charts.
